# Module 09 — Text Classification with an Embedding + LSTM (TensorFlow)

**Goal:** the TensorFlow counterpart to Module 05 — classify news
headlines into 4 topics (World, Sports, Business, Sci/Tech) with a
word-embedding + LSTM classifier, using Keras's `TextVectorization` layer
to handle tokenization for you.

**Before you start:** launch with a **TensorFlow (GPU)** kernel (see this
module's [README](README.md)).

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
assert len(gpus) > 0, (
    "No GPU detected in this kernel.\n"
    "Go back to the portal's Servers tab, stop this session, spawn a new one "
    "using the kernel named in this module's README, then run this again."
)
print("GPU(s):", gpus)

## 0. Install the dataset library

In [ ]:
%pip install -q datasets

## 1. Load the AG News dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset("ag_news")
label_names = raw["train"].features["label"].names
print(label_names)
print(raw["train"][0])

train_texts, train_labels = raw["train"]["text"], raw["train"]["label"]
test_texts, test_labels = raw["test"]["text"], raw["test"]["label"]

## 2. Build a text vectorization layer

In [ ]:
max_len = 40
vocab_size = 20000

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=max_len,
)
vectorize_layer.adapt(train_texts)

print(f"Vocabulary size: {len(vectorize_layer.get_vocabulary())}")

## 3. Build tf.data datasets

In [ ]:
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_texts, train_labels))
    .shuffle(10000)
    .batch(128)
    .map(lambda x, y: (vectorize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((test_texts, test_labels))
    .batch(256)
    .map(lambda x, y: (vectorize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)

## 4. Define the model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(max_len,)),
    tf.keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(4),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

## 5. Train

In [ ]:
history = model.fit(train_ds, epochs=4, validation_data=test_ds)

## 6. Try it on your own headline

In [ ]:
def predict(text):
    logits = model.predict(vectorize_layer(tf.constant([text])), verbose=0)
    return label_names[logits.argmax(axis=1)[0]]

for headline in [
    "Local team wins championship after dramatic overtime finish",
    "Central bank raises interest rates amid inflation concerns",
    "New chip promises twice the performance at half the power",
]:
    print(f"{headline!r} -> {predict(headline)}")

## Exercises

1. **Bidirectional LSTM.** Wrap the LSTM in
   `tf.keras.layers.Bidirectional(...)` — does accuracy improve?
2. **Try GRU instead of LSTM** (`tf.keras.layers.GRU`, same interface) —
   does it train faster? Does accuracy change?
3. **Vocabulary size.** Try `vocab_size = 5000` and `vocab_size = 50000`
   — how does that affect accuracy and embedding-layer memory use?
4. **Stretch goal:** plot a confusion matrix
   (`sklearn.metrics.confusion_matrix`) over the 4 classes, and compare
   directly with Module 05's PyTorch version — same task, same dataset,
   different framework.

In [ ]:
# Your experiments here
